# More Datasets
This notebooks investigates further datasets

In [ ]:
# Setting everything to be deterministic
%load_ext autoreload
%autoreload 2
print(__debug__)

import logging
logging.basicConfig(level=logging.INFO)

import seaborn as sns
sns.set_theme(style="whitegrid")


import numpy as np
import random, torch, os, cv2

seed = 0

print(f"Using device: {torch.device("cuda" if torch.cuda.is_available() else "cpu")}")

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
cv2.setRNGSeed(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ["PYTHONHASHSEED"] = str(seed)

import matplotlib.pyplot as plt

# Import the headset_localization package (used for localizers)
from headset_localization import *
# Import the CompleteRobotScan from the data generation pipeline
from shared import CompleteRobotScan

In [ ]:
round1_bright_scan_loc = "../example_datasets/round1_bright"
round1_medium_scan_loc = "../example_datasets/round1_medium"


round1_medium_1_vrs_loc = "../example_datasets/round1_medium_1.vrs"
round1_medium_2_vrs_loc = "../example_datasets/round1_medium_2.vrs"
round1_medium_3_vrs_loc = "../example_datasets/round1_medium_3.vrs"


round1_bright_scan = CompleteRobotScan.from_folder(round1_bright_scan_loc)
round1_medium_scan = CompleteRobotScan.from_folder(round1_medium_scan_loc)

round1_bright_env = Scanned3dEnvironment.from_gathered_robot_data(
        round1_bright_scan, number_of_sampled_datapoints=100,
        est3d_xyz_image_gen_config = XYZImageGenerationConfig(iforest_contamination=0.05, use_depth_images_if_provided=True),
        est3d_xyz_icp_config=ICPAlignmentConfig()
)

round1_medium_env = Scanned3dEnvironment.from_gathered_robot_data(
        round1_medium_scan, number_of_sampled_datapoints=100,
        est3d_xyz_image_gen_config = XYZImageGenerationConfig(iforest_contamination=0.05, use_depth_images_if_provided=True),
        est3d_xyz_icp_config=ICPAlignmentConfig()
)

round1_medium_1_hr = HeadsetRecording.from_folder("../example_datasets/round1_medium_rec1")
round1_medium_2_hr = HeadsetRecording.from_folder("../example_datasets/round1_medium_rec2")
round1_medium_3_hr = HeadsetRecording.from_folder("../example_datasets/round1_medium_rec3")

In [ ]:
visualize_loaded_data = True

if visualize_loaded_data:
    for headset_data in [round1_medium_1_hr, round1_medium_2_hr, round1_medium_3_hr]:
        visualize_robot_camera_environment_combo(robot_env=round1_medium_env, headset_data=headset_data)

chosen_headset_data = round1_medium_2_hr

In [ ]:
from headset_localization import *

# Creation of the Predictors
points_light_glue = GradableLocalizer(
    creator= PnPLocalizer.get_creation_function(
        cam2_intrinsic_mtx=chosen_headset_data.intrinsic_cam_mtx,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            extract_and_match=ExtractAndLightGlue(),
            crop_augmentations=[0.4]
        )
    ),
    name="PnP-LG"
)

points_loma = GradableLocalizer(
    creator= PnPLocalizer.get_creation_function(
        cam2_intrinsic_mtx=chosen_headset_data.intrinsic_cam_mtx,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            extract_and_match=ExtractAndMatchLoMa('LoMaB128'),
            crop_augmentations=[0.2]
        )
    ),
    name="PnP-LoMa"
)

points_lines_light_glue = GradableLocalizer(
    creator=PnPLLocalizer.get_creation_function(
        cam2_intrinsic_mtx = chosen_headset_data.intrinsic_cam_mtx,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            extract_and_match=ExtractAndLightGlue(),
            crop_augmentations=[0.4]
        ),
        cam1_line_generator = LineGenerator(lsd_diagonal_size=850),
    ),
    name="PnP+L-LG"
)

yolo = YOLOv26Segmenter("yoloe-26l-seg.pt", prompts=["cup", "fruit", "plate", "teddy", "ball", "tennisball", "pen", "lego", "brick", "duplo"])

ellipsoids_light_glue = GradableLocalizer(
    creator=EllipsoidLocalizer.get_creation_function(
        cam2_intrinsic_mtx = chosen_headset_data.intrinsic_cam_mtx,
        matching_config=GaussianMatchingConfig(dummy_value=0.001),
        cam1_segmenter = SAM3Segmenter(Sam3Prompt(mask_threshold=0.2)),
        ellipsoid_fitter = MVEEEllipsoidFitter(contamination=0.2),
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            extract_and_match=ExtractAndLightGlue(),
            crop_augmentations=[0.4],
        ),
        ellipsoid_matching_config=PointCloudMatchingConfig(min_cluster_size=2),
        visualize_environment_generation = True
    ),
    name="Ellipsoids"
)

In [ ]:
grader_medium = NPredictors1DatasetGrader(
    gradable_pose_predictors=[points_light_glue, points_loma, ellipsoids_light_glue, points_lines_light_glue],
    headset_data = chosen_headset_data,
    robot_env = round1_medium_env,
    compute_ray_intersection_error=True, use_tqdm_for_frames=False, use_tqdm_for_predictors=True
)

In [ ]:
if False:
    grader.visualize_predictions_3d()

grader_medium.print_summary()

fig1, ax = plt.subplots(1, 1, figsize = (12, 3))
grader_medium.plot_time_series_error(ax, TimeSeriesErrorType.ABS_TRANSLATIONAL)

fig1, axes = plt.subplots(2, 3, figsize = (15, 8))
grader_medium.plot_signed_error_comparison(axes = [axes[0,0], axes[0,1], axes[0,2], axes[1,0], axes[1,1], axes[1,2]], explain = True)

In [ ]:
grader_bright = NPredictors1DatasetGrader(
    gradable_pose_predictors=[points_light_glue, points_loma, ellipsoids_light_glue, points_lines_light_glue],
    headset_data = chosen_headset_data,
    robot_env = round1_bright_env,
    compute_ray_intersection_error=True, use_tqdm_for_frames=False, use_tqdm_for_predictors=True
)

In [ ]:
if False:
    grader.visualize_predictions_3d()

grader_bright.print_summary()

fig1, ax = plt.subplots(1, 1, figsize = (12, 3))
grader_bright.plot_time_series_error(ax, TimeSeriesErrorType.ABS_TRANSLATIONAL)

fig1, axes = plt.subplots(2, 3, figsize = (15, 8))
grader_bright.plot_signed_error_comparison(axes = [axes[0,0], axes[0,1], axes[0,2], axes[1,0], axes[1,1], axes[1,2]], explain = True)